In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
print("Libraries imported")

Libraries imported


In [3]:
df = pd.read_csv("data/processed/cleaned_train.csv")
print(df.shape)
df.head()

C:\Users\Soumyajit\AppData\Local\Temp\ipykernel_4592\2326169302.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/processed/cleaned_train.csv")


(844338, 18)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,0.0,0.0,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,0.0,0.0,NaN


date conversion

In [4]:
df["Date"] = pd.to_datetime(df["Date"])
print(df["Date"].dtype)

datetime64[ns]


In [5]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
df["DayOfWeekNum"] = df["Date"].dt.dayofweek
df["IsWeekend"] = df["DayOfWeekNum"].isin([5, 6]).astype(int)
df["IsMonthStart"] = df["Date"].dt.is_month_start.astype(int)
df["IsMonthEnd"] = df["Date"].dt.is_month_end.astype(int)

df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Year,Month,Day,WeekOfYear,DayOfWeekNum,IsWeekend,IsMonthStart,IsMonthEnd
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,0.0,0.0,NaN,2015,7,31,31,4,0,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",2015,7,31,31,4,0,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",2015,7,31,31,4,0,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,0.0,0.0,NaN,2015,7,31,31,4,0,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,0.0,0.0,NaN,2015,7,31,31,4,0,0,1


In [6]:
df["CompetitionOpenYears"] = df["Year"] - df["CompetitionOpenSinceYear"]
df["CompetitionOpenYears"] = df["CompetitionOpenYears"].apply(lambda x: x if x > 0 else 0)

In [7]:
df["Promo2RunningYears"] = df["Year"] - df["Promo2SinceYear"]
df["Promo2RunningYears"] = df["Promo2RunningYears"].apply(lambda x: x if x > 0 else 0)

In [8]:
selected_columns = [
    "Store",
    "DayOfWeek",
    "Promo",
    "SchoolHoliday",
    "StateHoliday",
    "StoreType",
    "Assortment",
    "CompetitionDistance",
    "Promo2",
    "Year",
    "Month",
    "Day",
    "WeekOfYear",
    "DayOfWeekNum",
    "IsWeekend",
    "IsMonthStart",
    "IsMonthEnd",
    "CompetitionOpenYears",
    "Promo2RunningYears",
    "Sales"
]

df = df[selected_columns]
print(df.shape)
df.head()

(844338, 20)


,Store,DayOfWeek,Promo,SchoolHoliday,StateHoliday,StoreType,Assortment,CompetitionDistance,Promo2,Year,Month,Day,WeekOfYear,DayOfWeekNum,IsWeekend,IsMonthStart,IsMonthEnd,CompetitionOpenYears,Promo2RunningYears,Sales
0,1,5,1,1,0,c,a,1270.0,0,2015,7,31,31,4,0,0,1,7.0,2015.0,5263
1,2,5,1,1,0,a,a,570.0,1,2015,7,31,31,4,0,0,1,8.0,5.0,6064
2,3,5,1,1,0,a,a,14130.0,1,2015,7,31,31,4,0,0,1,9.0,4.0,8314
3,4,5,1,1,0,c,c,620.0,0,2015,7,31,31,4,0,0,1,6.0,2015.0,13995
4,5,5,1,1,0,a,a,29910.0,0,2015,7,31,31,4,0,0,1,0.0,2015.0,4822


one hot encoding

In [9]:
df = pd.get_dummies(
    df,
    columns=["StateHoliday","StoreType","Assortment"],
    drop_first = True
)
print(df.shape)
df.head()

(844338, 26)


,Store,DayOfWeek,Promo,SchoolHoliday,CompetitionDistance,Promo2,Year,Month,Day,WeekOfYear,DayOfWeekNum,IsWeekend,IsMonthStart,IsMonthEnd,CompetitionOpenYears,Promo2RunningYears,Sales,StateHoliday_0,StateHoliday_a,StateHoliday_b,StateHoliday_c,StoreType_b,StoreType_c,StoreType_d,Assortment_b,Assortment_c
0,1,5,1,1,1270.0,0,2015,7,31,31,4,0,0,1,7.0,2015.0,5263,False,False,False,False,False,True,False,False,False
1,2,5,1,1,570.0,1,2015,7,31,31,4,0,0,1,8.0,5.0,6064,False,False,False,False,False,False,False,False,False
2,3,5,1,1,14130.0,1,2015,7,31,31,4,0,0,1,9.0,4.0,8314,False,False,False,False,False,False,False,False,False
3,4,5,1,1,620.0,0,2015,7,31,31,4,0,0,1,6.0,2015.0,13995,False,False,False,False,False,True,False,False,True
4,5,5,1,1,29910.0,0,2015,7,31,31,4,0,0,1,0.0,2015.0,4822,False,False,False,False,False,False,False,False,False


In [10]:
df.to_csv("data/processed/featured_train.csv", index=False)
print("Feature-engineered data saved to /data/processed/featured_train.csv")

Feature-engineered data saved to /data/processed/featured_train.csv
